# 01. OpenAlex API アクセス入門

## このノートブックでやること
科学計量学（science of science）の第一歩は「**論文データを手に入れること**」です。このノートブックは**チュートリアル全体のデータ取得の入口**であり、同時に **[OpenAlex](https://openalex.org/) API の使い方そのものを学ぶ**チュートリアルです。

無料の学術データベース OpenAlex の REST API から、論文（Works）・著者（Authors）・研究機関（Institutions）・雑誌（Sources）などを、**検索・絞り込み・集計・ページング**しながら取得する方法を一通り体験します。最後に、あとで使いやすいように **2つの表（pandas DataFrame）** に整えて保存します。

### このノートブックが保存するデータ（02以降が「読むだけ」で済むように）
| ファイル | 粒度 | 主な利用先 |
|---|---|---|
| `data/works/{NAME}/works.parquet` | **1行 = 1論文** | 02 基礎統計 / 04 ネットワーク |
| `data/works/{NAME}/paper_authors.parquet` | **1行 = 1著者 × 1論文** | 03 研究者分析 / 05 ジェンダー |
| `data/career/paper_authors.parquet`（付録） | ランダム著者の全業績 | 03 §3-2 ランダムインパクトルール |

> **02〜06 のノートブックは、これらのファイルを `pd.read_parquet(...)` で読むだけ**です。面倒なAPI処理はすべてこの01に集約してあります。**まずこの01を実行**してデータを用意してから、02以降に進んでください（事前計算済みデータは自動ダウンロードもできます）。

## 科学計量学（science of science）と OpenAlex
科学そのものを「論文・著者・機関・助成金・雑誌」というエンティティのネットワークとして定量的に分析する分野を **science of science / 科学計量学（scientometrics）** と呼びます。データには大きく2種類あります。

- **商用データベース**: Web of Science (Clarivate)、Scopus (Elsevier)。品質は高いが有償。
- **オープンアクセス DB**: 誰でも無償でAPIから取得できる。**OpenAlex**、Crossref、Semantic Scholar、PubMed など。

本チュートリアルは**完全無償・APIが使いやすい [OpenAlex](https://docs.openalex.org/)** を使います。OpenAlex は旧 Microsoft Academic Graph (MAG) の後継として2022年に公開され、現在は3億件を超える論文を収録しています（件数は日々増えます）。

### OpenAlex の主なエンティティ
| エンティティ | 内容 | エンドポイント |
|---|---|---|
| **Works** | 論文・書籍・データセット等の成果物 | `/works` |
| **Authors** | 著者 | `/authors` |
| **Sources** | 雑誌・リポジトリ等の発行元 | `/sources` |
| **Institutions** | 研究機関 | `/institutions` |
| **Topics** | 研究テーマ（4階層） | `/topics` |
| **Funders** | 助成機関 | `/funders` |

### 題材：バクテリオファージ研究
本チュートリアルは一貫して **1つの研究テーマ** に絞って分析します。題材は OpenAlex のトピック **`T11048` "Bacteriophages and microbial interactions"**（バクテリオファージ＝細菌に感染するウイルスと微生物間相互作用の研究）です。冒頭の `NAME` を `'qc'` にすると量子情報（`T10020`）に丸ごと切り替わります。

> **実装メモ**: 依存を増やさないため `requests` で直接APIを叩きます。リクエストに `mailto`（連絡先メール）を付けると "polite pool" に入り、安定して速く取得できます。**認証キーは不要**です。

## 準備：実行環境のセットアップ
次のセルは、最初に一度だけ実行する「おまじない」です。**このチュートリアルを動かすのに必要な準備**を自動で整えます。中身を細かく理解する必要はありません。

- **ローカル（自分のPCの Jupyter）**: 作業フォルダをプロジェクトのルート（`data/` がある場所）に合わせるだけ。
- **Google Colab**: 必要なライブラリのインストールと、GitHub からのコード一式のダウンロードまで自動で行います。

あわせて `ensure_data()` という関数を定義します。これは「**事前計算済みのデータが手元に無ければ GitHub から自動ダウンロードして展開する**」ための道具で、あとのセルから呼び出します。この段階ではまだ何も取得・分析しないので、通常は出力はありません。

In [1]:
# === セットアップ（このセルを最初に実行）===
# ローカル(Jupyter): 依存の確認と作業ディレクトリの調整のみ。
# Google Colab   : 依存インストール＋リポジトリ取得も自動で行う。
import os, sys

# データを変更して他ノートと共有・永続化したい人だけ True に（Google Drive をマウントします）。
# False（既定）なら使い捨てランタイム内で完結し、必要データは GitHub Release から取得します（権限プロンプト不要・完全ワンクリック）。
USE_DRIVE = False

if 'google.colab' in sys.modules:
    !pip -q install requests pandas pyarrow numpy matplotlib scikit-learn scipy powerlaw networkx igraph leidenalg umap-learn gender-guesser iso4 nltk openai
    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')                      # Trueにした人だけ Drive 権限を承認（各自のDrive・同名でOK）
        BASE = '/content/drive/MyDrive/sciscitutorial'     # ノート間・セッション間でデータを永続共有
    else:
        BASE = '/content/sciscitutorial'                   # 使い捨てランタイム内。権限不要（データはRelease/APIから）
    if not os.path.exists(f'{BASE}/.git'):
        !git clone -q https://github.com/asatani/sciscitutorial.git {BASE}
    os.chdir(BASE)

# code/ から起動した場合はプロジェクトルート（data/ がある場所）へ移動する。
if os.path.basename(os.getcwd()) == 'code':
    os.chdir('..')

# 事前計算済みデータが無ければ GitHub Release から取得し data/ 以下に展開する。
# works=対象トピックの論文, career=03 §3-2 用のランダム著者, supplementary=06(DI)用のエッジリスト。
def ensure_data(name, works=False, career=False, supplementary=False):
    import urllib.request, zipfile
    RELEASE = 'https://github.com/asatani/sciscitutorial/releases/download/data-v1'
    needs = []
    if works:         needs.append((f'data/works/{name}', f'works_{name}.zip'))
    if career:        needs.append(('data/career', 'career.zip'))
    if supplementary: needs.append(('data/supplementary', 'supplementary.zip'))
    os.makedirs('data', exist_ok=True)
    for path, asset in needs:
        if os.path.exists(path):
            continue
        print('downloading', asset, '...')
        zip_path = f'data/{asset}'
        urllib.request.urlretrieve(f'{RELEASE}/{asset}', zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall('data')
        os.remove(zip_path)

## この分析で使う「共通の道具」をそろえる
次のセルでは、以降のすべてのセルが使う **設定と道具（ヘルパー関数）** を用意します。ポイントは3つです。

1. **どの研究テーマを分析するか決める**（`NAME`）。既定は `'bacteria'`（バクテリオファージ研究＝トピック `T11048`）。ここを `'qc'` にすると量子情報のデータセットに丸ごと切り替わり、以降の分析がそのまま別テーマで走ります。
2. **保存先フォルダを決める**（取得データは `data/works/{NAME}/`、図表は `output/{NAME}/`）。
3. **OpenAlex API を叩く共通関数 `oa_get()` を定義する**。以降 API を呼ぶときは毎回 URL を組み立てる代わりに、この関数ひとつを使い回します。連絡先メール（`mailto`）を自動で付けるので、"polite pool"（行儀のよい利用者向けの速い窓口）に入って安定して取得できます。**認証キーは不要**です。

このセルもまだデータ取得はしません。準備だけなので出力はありません。

In [2]:
import os, time, json
import requests
import numpy as np
import pandas as pd

# --- データセット選択（'bacteria'=T11048 バクテリオファージ / 'qc'=T10020 量子情報）---
NAME       = 'bacteria'                                    # ← 'qc' に変えれば量子データセットに切替
DATASETS   = {'bacteria': ('T11048', 'Bacteriophages and microbial interactions'),
              'qc':       ('T10020', 'Quantum Information and Cryptography')}
TOPIC, TOPIC_NAME = DATASETS[NAME]                        # NAME からトピックID・名前を決定
DATA_DIR   = f'data/works/{NAME}'                         # 取得データの保存先
OUT_DIR    = f'output/{NAME}'                             # 図表の保存先
MAILTO     = 'asatani@gmail.com'                          # APIに付ける連絡先（polite pool で安定・高速化）

ensure_data(NAME, works=True)                             # 既存の事前計算済みデータがあれば取得（無ければ後半でAPIから自作）
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
pd.set_option('display.max_columns', 100)

# --- 全エンドポイント共通のヘルパー：JSONを返し、mailto を自動付与する ---
BASE = 'https://api.openalex.org'
def oa_get(entity, **params):
    """OpenAlex の任意エンドポイント（'works','authors',... や 'works/W123'）を叩いてJSONを返す。
    キーワードに '-' が使えないので per_page= を per-page に読み替える。"""
    if 'per_page' in params:
        params['per-page'] = params.pop('per_page')
    params.setdefault('mailto', MAILTO)
    r = requests.get(f'{BASE}/{entity}', params=params, timeout=60)
    r.raise_for_status()
    return r.json()

## 0. OpenAlex の全体像（規模とトピック階層）
個別のトピックに潜る前に、OpenAlex が**どれくらいの規模**で、論文が**どんな分類体系**で整理されているかを把握します。`per-page=1` で問い合わせると、結果本体を受け取らずに `meta.count`（該当件数）だけを安価に取得できます。

In [3]:
# 各エンティティの総数だけを安価に取る（per-page=1 → 本体を返さず meta.count だけ）。
def oa_count(entity, filter=None):
    """entity（'works' や 'authors' など）の該当件数だけを返す。"""
    params = {'per_page': 1}
    if filter:
        params['filter'] = filter
    return oa_get(entity, **params)['meta']['count']

try:
    # OpenAlex が持つ主要エンティティの総数を表示
    for entity in ['works', 'authors', 'sources', 'institutions', 'topics', 'funders']:
        print(f'{entity:13s}: {oa_count(entity):>14,}')

    # 対象トピックの階層（domain > field > subfield > topic）を確認する。
    topic = oa_get(f'topics/{TOPIC}')
    print(f'\nTopic {TOPIC}:', topic['display_name'])
    print('  domain   >', topic['domain']['display_name'])
    print('  field    >', topic['field']['display_name'])
    print('  subfield >', topic['subfield']['display_name'])
    print(f"  works    : {topic['works_count']:,}")
except Exception as e:
    print('OpenAlex API error:', e)

works        :    318,688,308
authors      :    119,275,573
sources      :        284,015
institutions :        127,784
topics       :          4,516
funders      :         45,640

Topic T11048: Bacteriophages and microbial interactions
  domain   > Physical Sciences
  field    > Environmental Science
  subfield > Ecology
  works    : 142,034


### 規模とトピック階層の読み方
- OpenAlex は**3億件超の論文**を収録し、世界の学術出版を広くカバーします。著者・機関・雑誌・助成機関も併せて持つため、論文単体では見えない関係（共著・所属・資金）まで分析できます。
- 研究テーマは **4階層** で整理されています:

  `domain（4区分）> field（26区分）> subfield（約250）> topic（約4,500）`

  本チュートリアルの **`T11048` (Bacteriophages and microbial interactions)** はこの最下層の topic です（上の出力で domain=Physical Sciences / field=Environmental Science / subfield=Ecology と確認できます）。
- topic 単位で絞ると、分野横断のノイズを抑えつつ「ひとつの研究テーマ」を深く分析できます。

> 注: 上のトピック総数は OpenAlex がその時点で返すライブの全期間件数です。一方、後半で使う parquet は配布済みスナップショットまたは取得時点の固定データなので、API の現在件数とは一致しないことがあります。

## 1. まず1件だけ取得して「論文の構造」を見る
一括取得の前に、**Work（論文）1件のJSON**を眺めて、どんなフィールドがあるかを掴みます。OpenAlex の各エンティティは URL 末尾のIDで直接引けます（例: `/works/W2741809807`）。返ってくるのは深くネストした辞書です。

抄録は著作権上の理由で、そのままの文章ではなく **inverted index（語 → 出現位置のマップ）** で返ります。あとで使う `abstract_text()` で通常の文章に復元します。

In [4]:
# 対象トピックで最も引用された論文を1件取り、そのIDで直接引き直してフィールド構造を見る。
top = oa_get('works', filter=f'primary_topic.id:{TOPIC}', sort='cited_by_count:desc', per_page=1)['results'][0]
SAMPLE_ID = top['id'].rsplit('/', 1)[-1]

w = oa_get(f'works/{SAMPLE_ID}')                       # /works/W... で1件直接取得
print('id         :', SAMPLE_ID)
print('title      :', w['display_name'])
print('year       :', w['publication_year'], '/ type:', w['type'], '/ cited_by:', w['cited_by_count'])
print('top-level fields:')
print(' ', ', '.join(sorted(w.keys())))

# 抄録は inverted index（語 -> 出現位置リスト）で返るので、通常の文章に戻す。
def abstract_text(inverted_index):
    if not inverted_index:
        return ''
    word_at = {pos: word for word, positions in inverted_index.items() for pos in positions}
    return ' '.join(word_at[i] for i in sorted(word_at))

print('\nabstract   :', abstract_text(w.get('abstract_inverted_index'))[:280], '...')

id         : W2100837269
title      : Cleavage of Structural Proteins during the Assembly of the Head of Bacteriophage T4
year       : 1970 / type: article / cited_by: 251666
top-level fields:
  abstract_inverted_index, apc_list, apc_paid, authorships, awards, best_oa_location, biblio, citation_normalized_percentile, cited_by_count, cited_by_percentile_year, concepts, content_urls, corresponding_author_ids, corresponding_institution_ids, countries_distinct_count, counts_by_year, created_date, display_name, doi, funders, fwci, has_content, has_fulltext, id, ids, indexed_in, institutions, institutions_distinct_count, is_paratext, is_retracted, is_xpac, keywords, language, locations, locations_count, mesh, open_access, primary_location, primary_topic, publication_date, publication_year, referenced_works, referenced_works_count, related_works, sustainable_development_goals, title, topics, type, updated_date

abstract   :  ...


### 出力の読み方：論文1件の「中身」
- 取り出したのは、このトピックで**最も引用された論文**（被引用 約25万回）。1970年の Laemmli の論文で、タンパク質を分離する電気泳動法（SDS-PAGE）を確立した超・定番の実験手法です。生命科学のあらゆる分野で使われるため、桁違いの被引用数になっています。
- `top-level fields` を見ると、1件の Work が **数十のフィールド**を持つことがわかります。タイトルや出版年だけでなく、著者と所属（`authorships`）、参考文献（`referenced_works`）、被引用の伸び（`counts_by_year`）、オープンアクセス状況（`open_access`）、トピック分類（`primary_topic` / `topics`）などが1つの JSON に詰まっています。**次の §2 では、この中から分析に使うフィールドだけを選んで表に整えます。**
- この論文では `abstract` がほぼ空です。古い論文は抄録データが登録されていないことがあり、欠損は珍しくありません。抄録がある論文では、`abstract_inverted_index`（語→出現位置のマップ）を `abstract_text()` で通常の文章に復元できます。

## 2. Works（論文）をまとめて取得して保存する
ここが**このノートブックの心臓部**です。対象トピック `T11048` の論文をまとめて取得し、分析しやすい2つの表（pandas DataFrame）に整えて parquet 形式で保存します。02以降のノートブックは、この保存済みファイルを読むだけで動きます。

大量の論文を効率よく・公平に集めるため、OpenAlex API の基本パラメータを組み合わせます。ここで覚える4つは、以降のあらゆる取得で使い回す「型」です。

- **絞り込み `filter`**: `primary_topic.id:T11048` でトピックを限定。さらに `to_publication_date`（上限日）を指定して取得対象の終端を固定します。直近の論文は**まだ引用が積み上がっていない**ため、被引用数を比べると不利になりやすいからです。
- **並び順 `sort`**: `cited_by_count:desc`（被引用の多い順）。件数を絞って試すときも、影響の大きい代表的な論文から拾えます。
- **列の選択 `select`**: 欲しいフィールドだけ指定すると、通信量が減って取得が速くなります。
- **ページ送り `cursor`**: API は1回で最大200件しか返しません。返ってくる `next_cursor`（次ページへの栞）を辿って全件を集めます（`page=` 方式は1万件までなので、大量取得では `cursor` を使います）。

まず取得を行う関数 `fetch_works()` を定義し（次のセル）、その次のセルで実際に取得・整形して2つの parquet に保存します。`data/works/{NAME}/works.parquet` が既にあればそれを読み込み、無ければ API から取得します。手早く試すなら `MAX_WORKS` を小さく（例: 5000）してかまいません。

In [ ]:
# --- OpenAlexから論文をまとめて取得する ---
# APIは1回で最大200件しか返さないので、next_cursor を辿って全ページを集める（cursorページング）。
def fetch_works(topic=TOPIC, to_date='2025-12-31', max_records=200000):
    params = dict(
        # filter: 対象トピック かつ 出版日が to_date 以前（終端を固定して再現性を持たせる）
        filter=f'primary_topic.id:{topic},to_publication_date:{to_date}',
        sort='cited_by_count:desc',                    # 被引用の多い順
        per_page=200, cursor='*',
        select=('id,doi,display_name,publication_year,type,cited_by_count,fwci,'
                'open_access,primary_topic,authorships,referenced_works,abstract_inverted_index'),
    )
    works = []
    while len(works) < max_records:
        r = oa_get('works', **params)
        works += r['results']
        print(f'fetched {len(works)}', end='\r')
        cursor = r['meta'].get('next_cursor')
        if not cursor or not r['results']:             # 次ページが無ければ終了
            break
        params['cursor'] = cursor
    return works[:max_records]

In [6]:
# トピックの論文をまとめて取得する。手早く試すなら小さくしてよい（例: MAX_WORKS = 5000）。
MAX_WORKS = 200000

# works.parquet / paper_authors.parquet が無ければAPIから作る。既にあれば読むだけ（2回目以降は一瞬）。
if os.path.exists(f'{DATA_DIR}/works.parquet'):
    works_df = pd.read_parquet(f'{DATA_DIR}/works.parquet')
    paper_authors = pd.read_parquet(f'{DATA_DIR}/paper_authors.parquet')
else:
    works = fetch_works(max_records=MAX_WORKS)         # APIから論文のJSONリストを取得（数分かかる）

    # (A) 1行 = 1論文 のテーブル works_df。各論文の入れ子JSONを、分析しやすい平らな列に変換する。
    works_df = pd.DataFrame([{
        'id': w['id'], 'short_id': w['id'].rsplit('/', 1)[-1], 'title': w.get('display_name'),
        'publication_year': w.get('publication_year'), 'type': w.get('type'),
        'cited_by_count': w.get('cited_by_count', 0), 'fwci': w.get('fwci'),
        'is_oa': (w.get('open_access') or {}).get('is_oa'),          # open_access の入れ子から is_oa を安全に取り出す
        'primary_topic': (w.get('primary_topic') or {}).get('display_name'),
        # 著者数・関与国・所属機関は authorships（各著者の所属情報のリスト）から集計する。
        # 集合 {...} を使うのは、同一著者が複数の所属で重複計上されても「1人」と数えるため（重複除去）。
        'n_authors': len({(a.get('author') or {}).get('id') for a in (w.get('authorships') or []) if (a.get('author') or {}).get('id')}),
        'countries': sorted({c for a in (w.get('authorships') or []) for c in (a.get('countries') or [])}),
        'institution_ids': sorted({i['id'].rsplit('/', 1)[-1] for a in (w.get('authorships') or []) for i in (a.get('institutions') or []) if i.get('id')}),
        'referenced_works': w.get('referenced_works') or [],        # この論文が引用した文献IDのリスト（引用ネットワーク用）
        'n_refs': len(w.get('referenced_works') or []),
        'abstract': abstract_text(w.get('abstract_inverted_index')),  # inverted index を文章に復元
    } for w in works]).sort_values('cited_by_count', ascending=False).reset_index(drop=True)

    # (B) 1行 = 1著者 × 1論文 のテーブル paper_authors（著者単位の分析に使う）。
    #     `for w in works for a in (...)` の二重ループで、1本の論文を「著者の数だけ」行に展開する。
    #     末尾の if は author.id を持たない著者（匿名・データ欠損）を除外する。
    paper_authors = pd.DataFrame([{
        'work_id': w['id'], 'title': w.get('display_name'),
        'publication_year': w.get('publication_year'),
        'cited_by_count': w.get('cited_by_count', 0), 'fwci': w.get('fwci'),   # 論文側の属性を各著者行にコピー（集計を高速化）
        'author_id': (a.get('author') or {}).get('id'),
        'author_name': (a.get('author') or {}).get('display_name'),
        'author_position': a.get('author_position'),                # first / middle / last（貢献度の目安）
        'institutions': [i['id'].rsplit('/', 1)[-1] for i in (a.get('institutions') or []) if i.get('id')],
        'countries': a.get('countries') or [],
    } for w in works for a in (w.get('authorships') or []) if (a.get('author') or {}).get('id')])

    works_df.to_parquet(f'{DATA_DIR}/works.parquet')              # 2つの表をディスクに保存 → 02以降はこれを読むだけ
    paper_authors.to_parquet(f'{DATA_DIR}/paper_authors.parquet')

print('works_df      :', works_df.shape)
print('paper_authors :', paper_authors.shape, '/ unique authors:', paper_authors['author_id'].nunique())

works_df      : (109897, 15)
paper_authors : (386167, 10) / unique authors: 192664


### 2-1. 保存したデータの列の意味（データ辞書）
02以降はこの2表を**読むだけ**なので、**各列が何を表し・どこで使うか**をここで明記します。

**`works.parquet`（1行 = 1論文）**
| 列 | 意味 | 主な利用先 |
|---|---|---|
| `id` / `short_id` | OpenAlex の論文ID（フル / 短縮） | 全ノートの突合キー |
| `title` | タイトル | 表示・テキスト分析 |
| `publication_year` | 出版年 | 年次推移（02, 04） |
| `type` | 文献タイプ（article 等） | タイプ別フィルタ（§8） |
| `cited_by_count` | 被引用数 | 引用の偏り・インパクト（02） |
| `fwci` | Field-Weighted Citation Impact（分野・年で正規化, 1.0=世界平均） | 公平なインパクト比較 |
| `is_oa` | オープンアクセスか | OA分析 |
| `primary_topic` | 主トピック名 | トピック分布 |
| `n_authors` | 著者数（重複著者は除外） | チームサイズ（02） |
| `countries` | 関与国コードのリスト | 国際共著（02 §4-1） |
| `institution_ids` | 関与機関の短縮IDリスト | 機関分析（04） |
| `referenced_works` | 参考文献の論文IDリスト | 引用ネットワーク（04） |
| `n_refs` | 参考文献数 | 引用の傾向 |
| `abstract` | 抄録（inverted index から復元） | 埋め込み・N-gram（02, 04） |

**`paper_authors.parquet`（1行 = 1著者 × 1論文）**
| 列 | 意味 | 主な利用先 |
|---|---|---|
| `work_id` | 論文ID | 論文との突合 |
| `title`, `publication_year`, `cited_by_count`, `fwci` | 論文側の属性（コピー） | 著者集計を高速化 |
| `author_id` / `author_name` | 著者ID / 氏名 | H-index・リーダーボード（03） |
| `author_position` | 著者の順序（first / middle / last） | 貢献度の目安 |
| `institutions` | その論文時点の所属機関ID | 機関別の研究者分析（03 §2-4） |
| `countries` | その著者の関与国 | 国別分析（03） |

> **注意（03で効いてくる重複）**: OpenAlex は所属機関ごとに同一著者を複数回計上することがあります。著者単位で集計するときは `drop_duplicates(['author_id','work_id'])` が必要です（放置すると h-index が論文数を超える等の不整合が出ます。詳細は03冒頭）。

### 2-2. 保存したデータの健全性チェック
分析に進む前に、取得したデータが**壊れていないか・偏っていないか**をざっと確認します。次のセルでは、出版年の範囲・被引用数の分布（最小/中央値/最大）・主要な列の欠損率を表示し、両テーブルの先頭数行を実際に眺めます。「まず自分のデータを見る」のはデータ分析の基本作法です。

In [7]:
# 件数・年範囲・被引用分布・欠損を見て、データが妥当か確認する。
print('year range:', int(works_df['publication_year'].min()), '-', int(works_df['publication_year'].max()))
print('citations : min %d / median %d / max %d' % (
    works_df['cited_by_count'].min(), works_df['cited_by_count'].median(), works_df['cited_by_count'].max()))
print('missing fwci/abstract:', works_df['fwci'].isna().mean().round(3), (works_df['abstract'] == '').mean().round(3))
display(works_df[['short_id', 'title', 'publication_year', 'cited_by_count', 'fwci', 'n_refs']].head())
display(paper_authors[['work_id', 'author_name', 'author_position', 'publication_year', 'cited_by_count']].head(3))

year range: 1753 - 2024
citations : min 0 / median 3 / max 251645
missing fwci/abstract: 0.132 0.403


,short_id,title,publication_year,cited_by_count,fwci,n_refs
0,W2100837269,Cleavage of Structural Proteins during the Ass...,1970,251645,276.1411,21
1,W2138270253,DNA sequencing with chain-terminating inhibitors,1977,69307,92.0422,12
2,W2028622989,Improved M13 phage cloning vectors and host st...,1985,15166,845.2060,38
3,W2019410656,A rapid alkaline extraction procedure for scre...,1979,14877,244.3710,20
4,W2128114769,Genome sequencing in microfabricated high-dens...,2005,7709,363.9373,21


,work_id,author_name,author_position,publication_year,cited_by_count
0,https://openalex.org/W2100837269,Ulrich K. Laemmli,first,1970,251645
1,https://openalex.org/W2138270253,Frederick Sanger,first,1977,69307
2,https://openalex.org/W2138270253,S. Nicklen,middle,1977,69307


### チェック結果の読み方
- **年の範囲が非常に広い**（例: 1753年〜2024年）: OpenAlex は古典的な文献まで収録しています。極端に古い年は件数がごく少なく外れ値になりやすいので、年次分析では必要に応じて期間を絞ります。
- **被引用数は激しく偏っている**: 中央値はわずか3回なのに、最大は25万回超。**大多数の論文はほとんど引用されず、ごく一部が桁違いに引用される**——これは科学計量学で繰り返し現れる「引用の強い偏り（べき乗則的な裾の重さ）」で、02で詳しく扱います。平均値ではなく中央値や分布で見るべき典型例です。
- **欠損は想定内**: `fwci` は約13%、`abstract` は約40%が欠損。古い論文や一部の文献タイプでは、これらの情報が登録されていないことがあります。欠損があること自体は正常で、分析時に該当行を除くなどして対処します。
- **先頭行の顔ぶれ**: 被引用順に並び、Laemmli(1970) や Sanger(1977, DNAシーケンシング) など**その分野を象徴する古典的名論文**が上位に来ていることが確認できます。データが正しく取れている良い兆候です。

## 3. キーワードで論文を検索する
IDが分からないときは**テキスト検索**で探します。OpenAlex には2通りあります。

- `search=...`（トップレベル）: タイトル・抄録・全文をまとめて検索し、**関連度順**で返す。
- `filter=title.search:...` / `abstract.search:...`: 対象を限定した検索。**他のフィルタ（年・被引用・OA など）と AND で組み合わせやすい**。

`filter` 内はカンマ `,` が AND、パイプ `|` が OR です。

In [8]:
# (1) タイトル・抄録・全文を横断検索（関連度順）
res = oa_get('works', search='phage therapy antibiotic resistance',
             per_page=5, select='id,display_name,publication_year,cited_by_count')
print('== search: "phage therapy antibiotic resistance" ==')
for w in res['results']:
    print(f"  {w['publication_year']}  cited={w['cited_by_count']:>5}  {w['display_name'][:70]}")

# (2) タイトル限定 + 追加フィルタ（2015年以降・被引用100超）を AND (,) で組み合わせる
res = oa_get('works',
             filter='title.search:CRISPR,from_publication_date:2015-01-01,cited_by_count:>100',
             sort='cited_by_count:desc', per_page=5,
             select='id,display_name,publication_year,cited_by_count')
print('\n== title:CRISPR / >=2015 / cited>100（被引用順）==')
for w in res['results']:
    print(f"  {w['publication_year']}  cited={w['cited_by_count']:>5}  {w['display_name'][:70]}")

== search: "phage therapy antibiotic resistance" ==
  2017  cited= 1108  Phage therapy: An alternative to antibiotics in the age of multi-drug 
  2019  cited= 5876  Antibiotic resistance threats in the United States, 2019
  2019  cited= 1189  Phage Therapy: A Renewed Approach to Combat Antibiotic-Resistant Bacte
  2019  cited=  984  Phage Therapy in the Postantibiotic Era
  2024  cited=  112  Phage therapy: A targeted approach to overcoming antibiotic resistance

== title:CRISPR / >=2015 / cited>100（被引用順）==
  2016  cited= 5002  Optimized sgRNA design to maximize activity and minimize off-target ef
  2015  cited= 4891  Cpf1 Is a Single RNA-Guided Endonuclease of a Class 2 CRISPR-Cas Syste
  2018  cited= 4382  CRISPR-Cas12a target binding unleashes indiscriminate single-stranded 
  2017  cited= 3834  Nucleic acid detection with CRISPR-Cas13a/C2c2
  2020  cited= 2804  CRISPR–Cas12-based detection of SARS-CoV-2


### 出力の読み方：2通りの検索の違い
- **(1) `search=`（横断検索）** は、キーワードに関連する論文を**関連度順**で返します。"phage therapy antibiotic resistance" では、ファージ療法を抗生物質耐性への対抗策とみなすレビュー論文が上位に来ます。**まず関連文献をざっと見つけたいとき**に向きます。
- **(2) `filter=title.search:` + 追加条件** は、タイトルに "CRISPR" を含む論文を、さらに「2015年以降」「被引用100超」で絞り込んだ結果です。カンマ `,` で条件を **AND** 連結できるのがポイントで、**年・引用数・国などの定量条件と組み合わせて母集団を厳密に作りたいとき**に向きます。
- 使い分けの目安: **探索的に見つけたい → `search`／条件を積んで正確に絞りたい → `filter`**。この2つを押さえれば、たいていの論文検索はカバーできます。

## 4. 著者を検索・取得する
著者名から著者エンティティを引き、その人の**全業績**を取得する流れです。同姓同名や表記ゆれがあるので、`works_count`・`cited_by_count`・所属で本人を見極めます。

- 検索: `/authors?search=氏名`
- 業績: `/works?filter=author.id:A...`（著者IDで論文を絞る）

In [9]:
# (1) 著者名で検索。候補を業績規模・所属とともに確認する。
res = oa_get('authors', search='Jennifer Doudna', per_page=5,
             select='id,display_name,works_count,cited_by_count,last_known_institutions')
print('== authors matching "Jennifer Doudna" ==')
for a in res['results']:
    inst = (a.get('last_known_institutions') or [{}])[0].get('display_name', '-')
    print(f"  {a['id'].rsplit('/',1)[-1]:>12}  works={a['works_count']:>4}  cited={a['cited_by_count']:>7}  {a['display_name']}  @ {inst}")

# (2) 先頭候補の著者IDで、その人の代表論文（被引用順）を取る
aid = res['results'][0]['id'].rsplit('/', 1)[-1]
works = oa_get('works', filter=f'author.id:{aid}', sort='cited_by_count:desc', per_page=5,
               select='id,display_name,publication_year,cited_by_count')
print(f'\n== top works of {aid} ==')
for w in works['results']:
    print(f"  {w['publication_year']}  cited={w['cited_by_count']:>6}  {w['display_name'][:70]}")

== authors matching "Jennifer Doudna" ==
   A5067184382  works= 670  cited= 119254  Jennifer A. Doudna  @ QB3
   A5084352079  works=  67  cited=   2819  Oscar Negrete  @ Sandia National Laboratories California
   A5130091406  works=   1  cited=     16  Jennifer A. Doudna  @ QB3
   A5128756606  works=   1  cited=      3  Jennifer A. Doudna  @ QB3
   A5129678894  works=   1  cited=      2  Jennifer A. Doudna  @ QB3

== top works of A5067184382 ==
  2012  cited= 17282  A Programmable Dual-RNA–Guided DNA Endonuclease in Adaptive Bacterial 
  2014  cited=  6959  The new frontier of genome engineering with CRISPR-Cas9
  2013  cited=  5177  Repurposing CRISPR as an RNA-Guided Platform for Sequence-Specific Con
  2018  cited=  4382  CRISPR-Cas12a target binding unleashes indiscriminate single-stranded 
  2013  cited=  3806  CRISPR-Mediated Modular RNA-Guided Regulation of Transcription in Euka


### 出力の読み方：同名著者の見分け方
- "Jennifer Doudna"（CRISPR 研究で 2020年ノーベル化学賞）で検索すると、**同姓同名・表記ゆれの候補が複数**出てきます。OpenAlex は名寄せ（同一人物の統合）を自動で行いますが完全ではなく、同じ人物が別IDに分裂していることもあります。
- 見分けの手がかりは **`works_count`（論文数）・`cited_by_count`（総被引用）・所属機関**。本人は業績規模が桁違いに大きく（論文670件・被引用約12万）、一目で特定できます。**著者分析では、集計の前に必ずこの本人確認を挟むこと**が重要です。
- 特定したIDで `filter=author.id:...` を使うと、その人の**全業績**を引けます。上位には CRISPR-Cas9 の記念碑的論文（2012, 被引用約1.7万）が並び、その研究者の代表作を一望できます。

## 5. 研究機関を検索・取得する
機関名から機関エンティティ（`I...`, [ROR](https://ror.org/) 紐付き）を引き、その機関の論文を取得します。04（分野ネットワーク）の機関別分析と同じ入口です。

- 検索: `/institutions?search=機関名`
- 論文: `/works?filter=institutions.id:I...`（対象トピックと AND で絞り込み）

In [10]:
# (1) 機関名で検索
res = oa_get('institutions', search='University of Tokyo', per_page=5,
             select='id,display_name,country_code,works_count,cited_by_count')
print('== institutions matching "University of Tokyo" ==')
for it in res['results']:
    print(f"  {it['id'].rsplit('/',1)[-1]:>10}  {it['country_code']}  works={it['works_count']:>8,}  {it['display_name']}")

# (2) 先頭機関 × 対象トピック の論文数と代表論文（institutions.id と primary_topic.id を AND）
iid = res['results'][0]['id'].rsplit('/', 1)[-1]
n = oa_count('works', filter=f'institutions.id:{iid},primary_topic.id:{TOPIC}')
print(f'\n{iid} × topic {TOPIC}: {n:,} works')
works = oa_get('works', filter=f'institutions.id:{iid},primary_topic.id:{TOPIC}',
               sort='cited_by_count:desc', per_page=5,
               select='id,display_name,publication_year,cited_by_count')
for w in works['results']:
    print(f"  {w['publication_year']}  cited={w['cited_by_count']:>5}  {w['display_name'][:66]}")

== institutions matching "University of Tokyo" ==
   I74801974  JP  works= 506,726  The University of Tokyo
  I161296585  JP  works=  91,696  Tokyo University of Science
  I4210109338  JP  works=  15,985  University of Tokyo Hospital
   I92614990  JP  works=  39,400  Tokyo University of Agriculture and Technology
  I4210117013  JP  works=   2,833  University of Tokyo Health Sciences

I74801974 × topic T11048: 521 works
  1997  cited= 3703  The complete genome sequence of the Gram-positive bacterium Bacill
  2007  cited=  350  Proposal of Lysinibacillus boronitolerans gen. nov. sp. nov., and 
  2002  cited=  249  Structure of a T7 RNA polymerase elongation complex at 2.9 Å resol
  1971  cited=  228  Role of Lipopolysaccharides in Antibiotic Resistance and Bacteriop
  1991  cited=  196  Abundance of Viruses in Marine Waters: Assessment by Epifluorescen


### 出力の読み方：機関の特定とトピックとの掛け合わせ
- "University of Tokyo" でも複数機関がヒットします（東京大学、東京理科大、東京農工大 …）。**名前が似た別機関**を取り違えないよう、`country_code` や `works_count`、正式名称で本命（`I74801974` The University of Tokyo）を見極めます。各機関は [ROR](https://ror.org/) という世界共通の機関IDにも紐づいています。
- 機関ID × トピックID を AND で掛け合わせると、「**その機関がこのテーマで出した論文**」だけに絞れます（東京大学 × バクテリオファージ = 521件）。所属の全論文ではなく特定テーマに限定できるのが、複数のエンティティを組み合わせる強みです。
- この「機関 × トピック」の絞り込みは、04（分野ネットワーク）の機関別分析の入口とまったく同じ考え方です。

## 6. 雑誌（Sources）の検索と、`group_by` による集計
**雑誌・リポジトリ（Sources）** も同じ流儀で検索できます。さらに OpenAlex の強力な機能が **`group_by`**：全件をダウンロードせずに、サーバ側で**カテゴリ別の件数集計**を返してくれます（年次推移や国別内訳を安価に得られます）。

In [11]:
# (1) 雑誌を名前で検索
res = oa_get('sources', search='Nature Microbiology', per_page=3,
             select='id,display_name,host_organization_name,works_count,type')
print('== sources matching "Nature Microbiology" ==')
for s in res['results']:
    print(f"  {s['id'].rsplit('/',1)[-1]:>10}  works={s['works_count']:>8,}  {s['display_name']}  ({s.get('type')})")

# (2) group_by: 対象トピックの「年次論文数」をサーバ側集計で取得（本体DL不要）
g = oa_get('works', filter=f'primary_topic.id:{TOPIC},to_publication_date:2024-12-31',
           group_by='publication_year')['group_by']
by_year = pd.DataFrame(g).astype({'key': int}).sort_values('key').tail(6)
print('\n== works per year（group_by・直近6年）==')
for _, r in by_year.iterrows():
    print(f"  {r['key']}: {r['count']:,}")

# (3) group_by: 関与国の内訳（上位5）
g = oa_get('works', filter=f'primary_topic.id:{TOPIC}',
           group_by='authorships.countries')['group_by'][:5]
print('\n== top countries in the topic（group_by）==')
for r in g:
    print(f"  {r['key_display_name']:>16}: {r['count']:,}")

== sources matching "Nature Microbiology" ==
  S2764926557  works=   2,839  Nature Microbiology  (journal)
   S56802129  works=   4,651  Nature Reviews Microbiology  (journal)

== works per year（group_by・直近6年）==
  2019: 3,651
  2020: 4,114
  2021: 4,154
  2022: 4,055
  2023: 4,284
  2024: 4,270

== top countries in the topic（group_by）==
  United States of America: 30,105
             China: 8,826
  United Kingdom of Great Britain and Northern Ireland: 6,416
           Germany: 5,368
            France: 4,925


### 出力の読み方：`group_by` でサーバ側集計
- (1) 雑誌検索は著者・機関と同じ流儀です。`Nature Microbiology` と `Nature Reviews Microbiology` のように紛らわしい名前も、掲載数やタイプ（journal など）で区別できます。
- (2)(3) が本節の主役 **`group_by`** です。**論文本体を1件もダウンロードせずに**、サーバ側で「年ごとの件数」「国ごとの件数」といった集計だけを返してもらえます。年次推移を見ると、このテーマの論文は年々増え、近年は年4,000件超で推移していることがわかります。国別では米国が突出し、中国・英国・独・仏が続きます。
- 全件取得は重く時間もかかりますが、**分布や推移を知りたいだけなら `group_by` が圧倒的に速く安価**です。保存済みデータを作る前の下調べや、API 上の最新件数を手早く確認したいときに便利です。

## 7. 便利機能：autocomplete と ランダムサンプリング
- **autocomplete**: 入力補完用の軽量検索。`/autocomplete/<entity>?q=...`。UIやID探索に便利です。
- **sample**: 条件を満たす母集団から**ランダム抽出**（`sample=N&seed=...`）。偏りのない標本が必要なとき（次の付録＝03 §3-2 のランダム著者など）に使います。`seed` を固定すれば再現できます。

In [12]:
# (1) autocomplete: 機関名の入力補完
print('== autocomplete/institutions q="max planck" ==')
for c in oa_get('autocomplete/institutions', q='max planck')['results'][:5]:
    print(f"  {c['id'].rsplit('/',1)[-1]:>10}  {c['display_name']}  ({c.get('hint','')})")

# (2) sample: 対象トピックからランダムに5件（seed で再現可能）
print('\n== random 5 works from the topic (seed=42) ==')
res = oa_get('works', filter=f'primary_topic.id:{TOPIC}', sample=5, seed=42, per_page=5,
             select='id,display_name,publication_year,cited_by_count')
for w in res['results']:
    print(f"  {w['publication_year']}  cited={w['cited_by_count']:>5}  {w['display_name'][:66]}")

== autocomplete/institutions q="max planck" ==
  I149899117  Max Planck Society  (Munich, Germany)
  I4210139716  Max Planck Institute for Plasma Physics  (Garching, Germany)
  I4210088365  Max Planck Institute for Solid State Research  (Stuttgart, Germany)
  I4210091772  Max Planck Institute for Extraterrestrial Physics  (Garching bei München, Germany)
  I4210109156  Max Planck Institute for Astronomy  (Heidelberg, Germany)

== random 5 works from the topic (seed=42) ==
  1986  cited=   25  Tobacco Mosaic Virus Mutants and Strains
  2020  cited=    0  Effect of Deletion of Gene Cluster involved in Synthesis of Entero
  2024  cited=    0  Experimental Phage Evolution Results in Expanded Host Ranges Again
  2016  cited=    0  The E.coli Sec Pathway under a Single-Molecule Loupe
  1986  cited=    0  A soft bacteriophage


### 出力の読み方：autocomplete と ランダムサンプリング
- **autocomplete** は入力補完向けの軽量検索です。"max planck" と打つだけで候補機関がIDつきで即座に返り、UI やID探しに便利です。
- **sample** は条件を満たす母集団からの**ランダム抽出**です。被引用順などで並べると上位の有名論文に偏りますが、`sample` は**偏りのない代表標本**を取れます。ランダムに引いた5件に被引用0の地味な論文が混じるのは、むしろ「大多数の論文の実像」を正しく映している証拠です。
- `seed=42` を固定しているので、**誰が実行しても同じ5件**が返り、結果を再現できます。この再現性は、次の付録（03 §3-2 用のランダム著者）でも重要になります。

## 8. フィルタ検索の例（年・国・型・引用数）

`filter` は条件をカンマで AND 連結でき、`group_by` を添えれば1回のAPIで分布も取れます。型（`type`）別の内訳と、年・国・被引用数を組み合わせた高被引用論文の取得例を示します。

In [13]:
# 文献タイプの分布（group_by をサーバ側集計で1回叩く）。
g = oa_get('works', filter=f'primary_topic.id:{TOPIC},to_publication_date:2024-12-31',
           group_by='type')['group_by']
display(pd.DataFrame(g)[['key', 'count']].head(8))

# 条件を組み合わせた例: 2020年以降・日本・被引用10超 の高被引用論文を取得。
r = oa_get('works',
           filter=f'primary_topic.id:{TOPIC},from_publication_date:2020-01-01,authorships.countries:JP,cited_by_count:>10',
           sort='cited_by_count:desc', per_page=10,
           select='display_name,publication_year,cited_by_count')['results']
display(pd.DataFrame([{'title': w['display_name'], 'year': w['publication_year'], 'cited': w['cited_by_count']} for w in r]))

,key,count
0,https://openalex.org/types/article,85086
1,https://openalex.org/types/review,4736
2,https://openalex.org/types/preprint,4336
3,https://openalex.org/types/book-chapter,4184
4,https://openalex.org/types/dataset,3623
5,https://openalex.org/types/dissertation,3016
6,https://openalex.org/types/paratext,1164
7,https://openalex.org/types/peer-review,704


,title,year,cited
0,Clades of huge phages from across Earth’s ecos...,2020,545
1,Targeted suppression of human IBD-associated g...,2022,514
2,Abolishment of morphology-based taxa and chang...,2023,469
3,Recent changes to virus taxonomy ratified by t...,2022,364
4,The gut virome: A new microbiome component in ...,2022,276
5,Taxonomy of prokaryotic viruses: 2018-2019upda...,2020,208
6,Microbial evolution through horizontal gene tr...,2024,207
7,Coevolutionary phage training leads to greater...,2021,171
8,"Isolation, characterization and application of...",2020,154
9,Biogeography of marine giant viruses reveals t...,2020,143


### 出力の読み方：フィルタの組み合わせ＋分布
- 上の表は文献タイプの内訳（`group_by='type'`）です。圧倒的多数が `article`（研究論文）で、review・preprint・book-chapter・dataset などが続きます。§2で `type` による絞り込みをするときの目安になります。
- 下の表は「**2020年以降・日本の関与・被引用10超**」を AND で重ねた高被引用論文です。このように `filter` に条件を積むだけで、「特定の国・期間・影響度」に対応する論文集合を1回の API 呼び出しで取り出せます。
- ここまでの要点は、**知りたい母集団を言葉から条件（filter）に翻訳する**こと。これが OpenAlex を使いこなす最大の鍵です。

## 付録：ランダム著者の全業績を取得する（03 §3-2 用）
03 §3-2 が読む `data/career/paper_authors.parquet` を OpenAlex から作るコードです。ポイントは、著者ごとに1回ずつ叩くのではなく、**著者IDの OR フィルタ（`author.id:A|B|…`, 最大50人/リクエスト）でまとめて取得**し、各 work の `authorships` を見て対象著者に振り分けること。これで数千リクエストが約100バッチ（5000人÷50人）に減ります。

重いので既定では `REBUILD_CAREERS = False`（実行しません）。作り直すときだけ `True` にしてください。通常はセットアップの `ensure_data(name, career=True)` で配布済みデータを使えば十分です。

In [14]:
REBUILD_CAREERS = False   # True にすると OpenAlex からランダム著者の全業績を作り直す（重い）

if REBUILD_CAREERS:
    DATA_PATH   = 'data/career/paper_authors.parquet'
    N_AUTHORS   = 5000                # ランダム抽出する著者数
    SAMPLE_SEED = 42                  # OpenAlex sample の乱数シード（再現用）
    MIN_WORKS   = 20                  # 抽出時の最低論文数
    BATCH       = 50                  # OR フィルタ（| 区切り）は最大50件/リクエスト
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

    session = requests.Session()
    session.headers.update({'User-Agent': f'sciscitutorial (mailto:{MAILTO})'})
    short_id = lambda url: url.rsplit('/', 1)[-1]

    def api_get(url, params):
        """200が返るまで指数バックオフで粘る。致命的なHTTPエラーのみ例外を投げる。"""
        wait = 5
        while True:
            try:
                resp = session.get(url, params={**params, 'mailto': MAILTO}, timeout=60)
            except requests.RequestException:
                time.sleep(wait); wait = min(wait * 2, 60); continue
            if resp.status_code == 200:
                return resp.json()
            if resp.status_code in (429, 500, 502, 503):
                retry_after = resp.headers.get('Retry-After')
                time.sleep(int(retry_after) if retry_after and retry_after.isdigit() else wait)
                wait = min(wait * 2, 60); continue
            raise RuntimeError(f'HTTP {resp.status_code}')

    def fetch_works_for_authors(short_ids):
        """複数著者(最大BATCH人)の全業績を author.id の OR でまとめて取得する。
        1リクエストで最大50人分の works を取り、各 work の authorships を見て対象著者に振り分ける
        （1本を複数の対象著者が共著していれば、それぞれの業績として計上）。"""
        want = set(short_ids)
        rows, cursor = [], '*'
        while cursor:
            page = api_get('https://api.openalex.org/works',
                           {'filter': f'author.id:{"|".join(short_ids)}', 'per-page': 200, 'cursor': cursor,
                            'select': 'id,publication_year,cited_by_count,fwci,authorships'})
            for w in page['results']:
                for a in (w.get('authorships') or []):
                    aid = short_id((a.get('author') or {}).get('id') or '')
                    if aid in want:
                        rows.append({'author_id': f'https://openalex.org/{aid}', 'work_id': w['id'],
                                     'publication_year': w.get('publication_year'),
                                     'cited_by_count': w.get('cited_by_count') or 0,
                                     'fwci': w.get('fwci')})
            cursor = page['meta'].get('next_cursor') if page['results'] else None
        return rows

    # (1) works_count>=20 の著者をランダム抽出する
    author_ids = []
    for page_no in range(1, N_AUTHORS // 200 + 2):
        page = api_get('https://api.openalex.org/authors',
                       {'filter': f'works_count:>{MIN_WORKS - 1}', 'sample': N_AUTHORS,
                        'seed': SAMPLE_SEED, 'per-page': 200, 'page': page_no, 'select': 'id'})
        author_ids += [a['id'] for a in page['results']]
        if len(page['results']) < 200:
            break
    author_ids = author_ids[:N_AUTHORS]
    print(f'ランダム著者: {len(author_ids)} 人')

    # (2) 50人ずつ author.id の OR でまとめて全業績を取得（250人ごとに途中保存・再開可能）
    if os.path.exists(DATA_PATH):
        careers = pd.read_parquet(DATA_PATH)
    else:
        careers = pd.DataFrame(columns=['author_id', 'work_id', 'publication_year', 'cited_by_count', 'fwci'])
    remaining = [a for a in author_ids if a not in set(careers['author_id'])]
    print(f'取得済み {careers["author_id"].nunique()} 人 / 残り {len(remaining)} 人')

    buffer = []
    for i in range(0, len(remaining), BATCH):
        chunk = [short_id(a) for a in remaining[i:i + BATCH]]
        try:
            buffer += fetch_works_for_authors(chunk)
        except RuntimeError:
            continue
        done = min(i + BATCH, len(remaining))
        if done % (BATCH * 5) < BATCH or done == len(remaining):
            careers = pd.concat([careers, pd.DataFrame(buffer)], ignore_index=True); buffer = []
            careers.to_parquet(DATA_PATH)
        print(f'fetched {done}/{len(remaining)} authors', end='\r')
        time.sleep(0.1)
    if buffer:
        careers = pd.concat([careers, pd.DataFrame(buffer)], ignore_index=True)
        careers.to_parquet(DATA_PATH)
    print(f'\n全業績: {careers.shape} | 著者数: {careers["author_id"].nunique()}')

## まとめ
- OpenAlex API を **単体取得 → キーワード検索 → 著者/機関/雑誌検索 → フィルタ&集計(`group_by`) → sample/autocomplete** と一通り体験しました。共通の作法は「**`filter` で絞り、`select` で列を選び、`cursor` で全件、`mailto` で polite pool**」です。
- チュートリアルの基盤として **`works.parquet`（1行 = 1論文）** と **`paper_authors.parquet`（1行 = 1著者 × 1論文）** を保存しました。**02以降はこれを読むだけ**です。
- 次は **02（基礎統計）** へ。取得したコーパスの規模・引用の偏り・国際共著などを見ていきます。